In [ ]:
# 실습용 csv 파일 준비

from pathlib import Path
import pandas as pd

csv_path = Path('sensor_log.csv')
sample = pd.DataFrame({
    'time': ['2026-08-24 09:00', '2026-08-24 09:01',
    '2026-08-24 09:02', '2026-08-24 09:03'],
    'sensor_id': ['S1', 'S1', 'S2', 'S2'],
    'temp': [25.0, None, 35.0, 42.0]
})

sample.to_csv(csv_path, index=False)    # 행 번호 없이 저장
print('저장 위치:', csv_path.resolve()) # 전체 경로 출력

저장 위치: C:\Users\User\OneDrive - kumoh\과제&수업자료\HINT\source\HINT\python\sensor_log.csv


In [ ]:
# 파일 확인과 자료형 변환

from pathlib import Path
import pandas as pd

def load_log(path):
    path = Path(path)
    
    if not path.exists():
        raise FileNotFoundError(
            f'파일이 없습니다: {path.resolve()}')
    
    df = pd.read_csv(path)
    required = {'time','sensor_id','temp'}
    missing = required - set(df.columns)
    
    if missing:
        raise ValueError(f'필수 열 누락: {sorted(missing)}')
    
    df['time'] = pd.to_datetime(df['time'], errors='coerce')    # 시간 변환
    df['temp'] = pd.to_numeric(df['temp'], errors='coerce')     # 온도 -> 숫자 변환
    
    return df

In [3]:
def clean_log(df):
    df = df.drop_duplicates().copy() # 중복 행 제거
    
    med = df.groupby('sensor_id')['temp'].transform('median')   # 센서별 중앙값
    df['temp'] = df['temp'].fillna(med)                         # 결측치를 센서별 중앙값
    
    overall = df['temp'].median()           # 전체 중앙값
    df['temp'] = df['temp'].fillna(overall) # 결측치를 중앙값으로
    
    needed = ['time', 'sensor_id', 'temp']  # 분석에 반드시 필요한 열
    df = df.dropna(subset=needed).copy()    # 필수값이 없는 행을 제거
    
    return df

In [ ]:
def flag_status(df, warn=30, danger=40):    # 상태 분류 함수
    df = df.copy()                          # 원본 복사본

    def classify(temp):     # 온도 분류 내부 함수
        if temp >= danger:
            return '위험'
        if temp >= warn:
            return '주의'
        return '정상'

    
    df['status'] = df['temp'].apply(classify) # 각 온도를 분류한다.
    return df

In [5]:
try:
    df = load_log('sensor_log.csv')
    df = clean_log(df)                  # 중복과 결측값 정제
    df = flag_status(df)                # 온도 상태 분류
    print(df.to_string(index=False))    # 행 번호 없이 출력
    
except (FileNotFoundError, ValueError) as error:
    print('실행 오류:', error)

               time sensor_id  temp status
2026-08-24 09:00:00        S1  25.0     정상
2026-08-24 09:01:00        S1  25.0     정상
2026-08-24 09:02:00        S2  35.0     주의
2026-08-24 09:03:00        S2  42.0     위험
